# Pet Emotion Classifier — Transfer Learning + Augmentation Robustness

**Mini-Hackathon #1: How Can Machines See What Matters?**

This notebook trains two ResNet18 models on a small pet-emotion dataset (4-class (angry / happy / sad / other)):
1. **Baseline** — minimal preprocessing only (resize + flip)
2. **Augmented** — augmentations tied to real-world photo variation (lighting, blur, rotation, occlusion, perspective)

Then we evaluate both on a *corrupted* held-out test set to show how thoughtful augmentation buys robustness, not just accuracy.

**Runtime:** GPU (T4) recommended. Total train + eval ≈ 5–10 min.

## 1. Clone the repo + install deps

In [ ]:
# Replace with your repo URL after you push to GitHub.
REPO_URL = 'https://github.com/YOUR-USERNAME/pet-emotion-hackathon.git'

import os
if not os.path.exists('pet-emotion-hackathon'):
    !git clone {REPO_URL}
%cd pet-emotion-hackathon
!pip install -q -r requirements.txt

## 2. Download the dataset from Kaggle

Dataset: [Pets Facial Expression Recognition](https://www.kaggle.com/datasets/anshtanwar/pets-facial-expression-dataset) by anshtanwar (~1000 images, 4 classes).

**Setup:** upload your `kaggle.json` (Kaggle account → Settings → API → Create New Token).

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json

In [ ]:
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d anshtanwar/pets-facial-expression-dataset -p data/ --unzip
!ls data/

In [ ]:
# The Kaggle archive nests the class folders one level deep. Flatten
# them into data/raw/<class>/ so our loader finds them.
import shutil
from pathlib import Path

root = Path('data')
target = root / 'raw'
target.mkdir(exist_ok=True)

for sub in root.rglob('*'):
    if sub.is_dir() and sub.name.lower() in {'happy', 'sad', 'angry', 'other'} and sub.parent != target:
        dest = target / sub.name
        if dest.exists():
            continue
        shutil.move(str(sub), str(dest))

for c in sorted(target.iterdir()):
    if c.is_dir():
        print(c.name, len(list(c.iterdir())))

## 3. Visualize the augmentations

Quick sanity check: do our augmentations look like real-world variation we'd expect to encounter?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from src.data import get_transforms, PetEmotionDataset, IMAGENET_MEAN, IMAGENET_STD

def denorm(t):
    t = t.clone()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

ds = PetEmotionDataset(root='data/raw', transform=None)
tx = get_transforms('augmented')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flat:
    img, label = ds[np.random.randint(len(ds))]
    aug = tx(img)
    ax.imshow(denorm(aug))
    ax.set_title(['angry', 'happy', 'sad', 'other'][label])
    ax.axis('off')
plt.suptitle('Random samples through the augmented pipeline')
plt.tight_layout()
plt.savefig('assets/aug_samples.png', dpi=110, bbox_inches='tight')
plt.show()

## 4. Train the baseline model (no thoughtful augmentation)

In [ ]:
from src.train import train

baseline_history = train(
    data_dir='data/raw',
    out_path='models/baseline.pt',
    train_mode='baseline',
    head_epochs=5,
    finetune_epochs=5,
)

## 5. Train the augmented model (same seed, same data, different transforms)

In [ ]:
augmented_history = train(
    data_dir='data/raw',
    out_path='models/augmented.pt',
    train_mode='augmented',
    head_epochs=5,
    finetune_epochs=5,
)

## 6. Robustness comparison

Evaluate both models on the same held-out test set with 6 corruption families × 5 severities.

In [ ]:
from src.evaluate import compare_models

results = compare_models(
    baseline_ckpt='models/baseline.pt',
    augmented_ckpt='models/augmented.pt',
    data_dir='data/raw',
)

import json
with open('assets/robustness_results.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# Plot: one panel per corruption family, x=severity, y=accuracy, two lines.
import matplotlib.pyplot as plt

corruptions = list(results['baseline'].keys())
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, name in zip(axes.flat, corruptions):
    sevs = sorted(int(k) for k in results['baseline'][name].keys())
    base_acc = [results['baseline'][name][s] for s in sevs]
    aug_acc  = [results['augmented'][name][s] for s in sevs]
    ax.plot(sevs, base_acc, marker='o', label='baseline')
    ax.plot(sevs, aug_acc, marker='s', label='augmented')
    ax.set_title(name)
    ax.set_xlabel('severity')
    ax.set_ylabel('accuracy')
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle('Robustness under image corruptions (held-out test set)')
plt.tight_layout()
plt.savefig('assets/robustness_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Headline summary numbers for the pitch:
import numpy as np
for model_name in ('baseline', 'augmented'):
    all_accs = [v for c in results[model_name].values() for v in c.values()]
    print(f'{model_name:>10s}  mean acc across all corruptions: {np.mean(all_accs):.3f}')

## 7. Download trained weights

Grab `models/augmented.pt` so you can upload it to your Hugging Face Space.

In [ ]:
from google.colab import files
files.download('models/augmented.pt')
files.download('assets/robustness_comparison.png')